Day 06
Project workflow
1. SQL: Create, inspect and retrieve data from the database.
2. pandas: Clean, transform and analyse the data.
3. Python: Write reusable functions to calculate business KPIs.
4. Final analysis: Answer business questions and make recommendations.

In [ ]:
%load_ext sql
%sql sqlite:///sales.db
    

In [12]:
import sqlite3
import pandas as pd
conn=sqlite3.connect("sales.db")

In [2]:
%%sql

DROP TABLE IF EXISTS ProjectOrders;
DROP TABLE IF EXISTS ProjectProducts;

CREATE TABLE ProjectProducts (
    product_id INTEGER,
    product_name TEXT,
    category TEXT,
    unit_price REAL
);

INSERT INTO ProjectProducts VALUES
(1, 'Laptop', 'Electronics', 800),
(2, 'Mouse', 'Electronics', 20),
(3, 'Keyboard', 'Electronics', 50),
(4, 'Desk', 'Furniture', 300),
(5, 'Chair', 'Furniture', 150);

CREATE TABLE ProjectOrders (
    order_id INTEGER,
    product_id INTEGER,
    quantity INTEGER,
    region TEXT,
    order_date TEXT
);

INSERT INTO ProjectOrders VALUES
(201, 1, 2, 'North', '2026-09-01'),
(202, 2, 5, 'South', '2026-09-02'),
(203, 3, 3, 'North', '2026-09-03'),
(204, 4, 2, 'West', '2026-09-04'),
(205, 2, 4, 'South', '2026-09-05'),
(206, 1, 1, 'East', '2026-09-06'),
(207, 5, 3, 'East', '2026-09-07'),
(208, 3, 6, 'West', '2026-09-08'),
(209, 4, 1, 'North', '2026-09-09'),
(210, 5, NULL, 'South', '2026-09-10'),
(211, 1, 2, 'West', '2026-09-11'),
(212, 2, 8, 'East', '2026-09-12');

 * sqlite:///sales.db
Done.
Done.
Done.
5 rows affected.
Done.
12 rows affected.


[]

In [5]:
%%sql
select*from ProjectOrders;

 * sqlite:///sales.db
Done.


order_id,product_id,quantity,region,order_date
201,1,2,North,2026-09-01
202,2,5,South,2026-09-02
203,3,3,North,2026-09-03
204,4,2,West,2026-09-04
205,2,4,South,2026-09-05
206,1,1,East,2026-09-06
207,5,3,East,2026-09-07
208,3,6,West,2026-09-08
209,4,1,North,2026-09-09
210,5,None,South,2026-09-10


In [4]:
%%sql
select*from ProjectProducts;

 * sqlite:///sales.db
Done.


product_id,product_name,category,unit_price
1,Laptop,Electronics,800.0
2,Mouse,Electronics,20.0
3,Keyboard,Electronics,50.0
4,Desk,Furniture,300.0
5,Chair,Furniture,150.0


In [10]:
%%sql
select order_id,order_date,product_name,category,quantity,unit_price,region,
    quantity*unit_price as revenue 
    from ProjectOrders join ProjectProducts
     on ProjectOrders.product_id= ProjectProducts.product_id;

 * sqlite:///sales.db
Done.


order_id,order_date,product_name,category,quantity,unit_price,region,revenue
201,2026-09-01,Laptop,Electronics,2,800.0,North,1600.0
202,2026-09-02,Mouse,Electronics,5,20.0,South,100.0
203,2026-09-03,Keyboard,Electronics,3,50.0,North,150.0
204,2026-09-04,Desk,Furniture,2,300.0,West,600.0
205,2026-09-05,Mouse,Electronics,4,20.0,South,80.0
206,2026-09-06,Laptop,Electronics,1,800.0,East,800.0
207,2026-09-07,Chair,Furniture,3,150.0,East,450.0
208,2026-09-08,Keyboard,Electronics,6,50.0,West,300.0
209,2026-09-09,Desk,Furniture,1,300.0,North,300.0
210,2026-09-10,Chair,Furniture,None,150.0,South,None


In [13]:
sql_query="""
select order_id,order_date,product_name,category,quantity,unit_price,region,
    quantity*unit_price as revenue 
    from ProjectOrders join ProjectProducts
     on ProjectOrders.product_id= ProjectProducts.product_id;
     """

In [14]:
sales_df=pd.read_sql_query(sql_query,conn)

In [15]:
sales_df.head()

,order_id,order_date,product_name,category,quantity,unit_price,region,revenue
0,201,2026-09-01,Laptop,Electronics,2.0,800.0,North,1600.0
1,202,2026-09-02,Mouse,Electronics,5.0,20.0,South,100.0
2,203,2026-09-03,Keyboard,Electronics,3.0,50.0,North,150.0
3,204,2026-09-04,Desk,Furniture,2.0,300.0,West,600.0
4,205,2026-09-05,Mouse,Electronics,4.0,20.0,South,80.0


In [18]:
sales_df.shape

(12, 8)

In [20]:
sales_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   order_id      12 non-null     int64  
 1   order_date    12 non-null     object 
 2   product_name  12 non-null     object 
 3   category      12 non-null     object 
 4   quantity      11 non-null     float64
 5   unit_price    12 non-null     float64
 6   region        12 non-null     object 
 7   revenue       11 non-null     float64
dtypes: float64(3), int64(1), object(4)
memory usage: 900.0+ bytes


In [22]:
sales_df.isna().sum()

order_id        0
order_date      0
product_name    0
category        0
quantity        1
unit_price      0
region          0
revenue         1
dtype: int64

In [23]:
sales_df[sales_df["quantity"].isna()]

,order_id,order_date,product_name,category,quantity,unit_price,region,revenue
9,210,2026-09-10,Chair,Furniture,NaN,150.0,South,NaN


In [25]:
sales_df["quantity"].describe().round(2)

count    11.00
mean      3.36
std       2.20
min       1.00
25%       2.00
50%       3.00
75%       4.50
max       8.00
Name: quantity, dtype: float64

In [26]:
sales_df.groupby("product_name")["quantity"].agg(["count", "mean", "median"])

,count,mean,median
product_name,,,
Chair,1,3.000000,3.0
Desk,2,1.500000,1.5
Keyboard,2,4.500000,4.5
Laptop,3,1.666667,2.0
Mouse,3,5.666667,5.0


In [31]:
sales_clean=sales_df.copy()
med_quan=sales_clean["quantity"].median()
sales_clean["quantity"]=sales_clean["quantity"].fillna(med_quan)
sales_clean["revenue"] = (sales_clean["quantity"] * sales_clean["unit_price"])
sales_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   order_id      12 non-null     int64  
 1   order_date    12 non-null     object 
 2   product_name  12 non-null     object 
 3   category      12 non-null     object 
 4   quantity      12 non-null     float64
 5   unit_price    12 non-null     float64
 6   region        12 non-null     object 
 7   revenue       12 non-null     float64
dtypes: float64(3), int64(1), object(4)
memory usage: 900.0+ bytes


In [33]:
sales_clean.isna().sum()

order_id        0
order_date      0
product_name    0
category        0
quantity        0
unit_price      0
region          0
revenue         0
dtype: int64

In [46]:
regional_revenue=sales_clean.groupby("region")["revenue"].sum().sort_values(ascending=False)
avg_order_regional_revenue=(regional_revenue/sales_clean.groupby("region")["order_id"].count()).round(2)


In [44]:
regional_revenue


region
West     2500.0
North    2050.0
East     1410.0
South     630.0
Name: revenue, dtype: float64

In [47]:
avg_order_regional_revenue

region
East     470.00
North    683.33
South    210.00
West     833.33
dtype: float64

In [53]:
sales_clean.groupby(["product_name"])[["quantity","revenue"]].sum().sort_values("revenue",ascending=False)

,quantity,revenue
product_name,,
Laptop,5.0,4000.0
Chair,6.0,900.0
Desk,3.0,900.0
Keyboard,9.0,450.0
Mouse,17.0,340.0


In [56]:
def calculate_total_revenue (df):
    total_revenue=round(df["revenue"].sum(),2)
    return total_revenue

In [57]:
calculate_total_revenue(sales_clean)

6590.0